# Train Final Deployment Model

Train the final deployment MobileNetV3-small model after official evaluation and write deployment-facing artifacts.

In [3]:
from __future__ import annotations

from meatlens_pork_pipeline.notebook_progress import (
    advance_notebook_cell_progress,
    finish_notebook_cell_progress,
    iter_notebook_progress,
    start_notebook_cell_progress,
)
NB_06_TRAIN_FINAL_DEPLOYMENT_MODEL_CELL_PROGRESS_1 = start_notebook_cell_progress('06_train_final_deployment_model.ipynb', 'Load shared training code', total_steps=1)

import json
import shutil
from pathlib import Path

import pandas as pd

shared_notebook = json.loads(Path('00_shared_setup.ipynb').read_text(encoding='utf-8'))
shared_code = '\n\n'.join(
    ''.join(cell.get('source', []))
    for cell in shared_notebook['cells']
    if cell.get('cell_type') == 'code'
)
exec(shared_code, globals())

training_notebook = json.loads(Path('04_train_8fold_mobilenetv3small.ipynb').read_text(encoding='utf-8'))
training_code_cells = [
    ''.join(cell.get('source', []))
    for cell in training_notebook['cells']
    if cell.get('cell_type') == 'code'
]
for code_source in training_code_cells[:3]:
    exec(code_source, globals())

finish_notebook_cell_progress(NB_06_TRAIN_FINAL_DEPLOYMENT_MODEL_CELL_PROGRESS_1)


[START] 06_train_final_deployment_model.ipynb | Load shared training code [0/1 step] elapsed=0.0s
[START] 00_shared_setup.ipynb | Shared setup bootstrap [0/1 step] elapsed=0.0s
Default processed dataset not ready at C:\Users\Adriaan M. Dimate\Desktop\development\school\meatlens-training-2\data\roboflow_processed_hsv_lab_threshold_roi_224 - run the early notebooks with raw or Excel input, or set overrides.
[RUNNING] 00_shared_setup.ipynb | Shared setup bootstrap | done [0/1 step] elapsed=0.0s
[RUNNING] 00_shared_setup.ipynb | Shared setup bootstrap | done [1/1 step] elapsed=0.0s
[START] 04_train_8fold_mobilenetv3small.ipynb | Load training dependencies [0/1 step] elapsed=0.0s
[START] 00_shared_setup.ipynb | Shared setup bootstrap [0/1 step] elapsed=0.0s
Default processed dataset not ready at C:\Users\Adriaan M. Dimate\Desktop\development\school\meatlens-training-2\data\roboflow_processed_hsv_lab_threshold_roi_224 - run the early notebooks with raw or Excel input, or set overrides.
[RUNN

In [4]:
from meatlens_pork_pipeline.notebook_progress import (
    advance_notebook_cell_progress,
    finish_notebook_cell_progress,
    iter_notebook_progress,
    start_notebook_cell_progress,
)
NB_06_TRAIN_FINAL_DEPLOYMENT_MODEL_CELL_PROGRESS_2 = start_notebook_cell_progress('06_train_final_deployment_model.ipynb', 'Define deployment helpers', total_steps=1)

def split_final_deployment_df(
    processed_df: pd.DataFrame,
    random_state: int,
    val_size: float,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    if DATASET_SOURCE == 'roboflow':
        train_path = GENERATED_SPLITS_ROOT / 'fold1_train.csv'
        val_path = GENERATED_SPLITS_ROOT / 'fold1_val.csv'
        if not train_path.is_file() or not val_path.is_file():
            raise FileNotFoundError('Run 03_build_cross_rotation_splits.ipynb first to generate the Roboflow 8-fold files.')
        train_df = pd.read_csv(train_path, dtype=str).fillna('')
        val_df = pd.read_csv(val_path, dtype=str).fillna('')
        return train_df.reset_index(drop=True), val_df.reset_index(drop=True)

    train_df, val_df = build_sample_heldout_validation_split(
        processed_df,
        val_size=val_size,
        random_state=random_state,
    )
    return train_df.reset_index(drop=True), val_df.reset_index(drop=True)


def train_final_deployment_model(processed_manifest_path: Path, output_root: Path) -> dict[str, object]:
    processed_df = pd.read_csv(processed_manifest_path, dtype=str).fillna('')
    train_df, val_df = split_final_deployment_df(
        processed_df,
        random_state=FINAL_TRAINING_SEED,
        val_size=FINAL_VAL_SIZE,
    )

    ensure_dir(output_root)
    ensure_dir(output_root / 'models')
    temp_train_csv = output_root / '_final_train.csv'
    temp_val_csv = output_root / '_final_val.csv'
    train_df.to_csv(temp_train_csv, index=False)
    val_df.to_csv(temp_val_csv, index=False)

    internal_fold_name = 'final'
    summary = train_single_fold(
        fold_name=internal_fold_name,
        seed=FINAL_TRAINING_SEED,
        train_csv=temp_train_csv,
        val_csv=temp_val_csv,
        test_csv=temp_val_csv,
        output_root=output_root,
        model_weights=MODEL_WEIGHTS,
        batch_size=BATCH_SIZE,
        epochs_head=EPOCHS_HEAD,
        epochs_fine=EPOCHS_FINE,
        use_training_augmentation=USE_TRAINING_AUGMENTATION,
    )

    final_model_path = output_root / 'models' / 'meatlens_final_8samples_cnn_only_mobilenetv3small.keras'
    final_predictions_path = output_root / 'final_validation_predictions.csv'
    final_history_path = output_root / 'final_training_history.csv'
    metadata_path = output_root / 'deployment_metadata.json'

    shutil.copy2(Path(str(summary['model_path'])), final_model_path)
    shutil.copy2(Path(str(summary['predictions_path'])), final_predictions_path)
    source_history_path = output_root / 'histories' / f'processed_roi8_cnn_only_{internal_fold_name}_seed{FINAL_TRAINING_SEED}_history.csv'
    shutil.copy2(source_history_path, final_history_path)

    metadata = {
        'model_name': 'meatlens_final_8samples_cnn_only_mobilenetv3small',
        'backbone': 'MobileNetV3Small',
        'model_input_mode': 'cnn_only',
        'image_crop_mode': INPUT_MODE,
        'input_mode': INPUT_MODE,
        'fine_tune_fraction': float(FINE_TUNE_FRACTION),
        'labels': LABEL_ORDER,
        'seed': FINAL_TRAINING_SEED,
        'validation_size': FINAL_VAL_SIZE,
        'train_count': len(train_df),
        'val_count': len(val_df),
        'accuracy': summary['accuracy'],
        'macro_precision': summary['macro_precision'],
        'macro_recall': summary['macro_recall'],
        'macro_f1': summary['macro_f1'],
        'model_path': str(final_model_path),
        'validation_predictions_path': str(final_predictions_path),
    }
    metadata_path.write_text(json.dumps(metadata, indent=2), encoding='utf-8')
    return {
        'model_path': final_model_path,
        'predictions_path': final_predictions_path,
        'history_path': final_history_path,
        'metadata_path': metadata_path,
    }

finish_notebook_cell_progress(NB_06_TRAIN_FINAL_DEPLOYMENT_MODEL_CELL_PROGRESS_2)


[START] 06_train_final_deployment_model.ipynb | Define deployment helpers [0/1 step] elapsed=0.0s
[RUNNING] 06_train_final_deployment_model.ipynb | Define deployment helpers | done [0/1 step] elapsed=0.0s
[RUNNING] 06_train_final_deployment_model.ipynb | Define deployment helpers | done [1/1 step] elapsed=0.0s


In [5]:
from meatlens_pork_pipeline.notebook_progress import (
    advance_notebook_cell_progress,
    finish_notebook_cell_progress,
    iter_notebook_progress,
    start_notebook_cell_progress,
)
NB_06_TRAIN_FINAL_DEPLOYMENT_MODEL_CELL_PROGRESS_3 = start_notebook_cell_progress('06_train_final_deployment_model.ipynb', 'Train final deployment model', total_steps=1)

RUN_SEEDS_LOCAL = [int(seed) for seed in override('RUN_SEEDS', RUN_SEEDS)]
FINAL_TRAINING_SEED = int(override('FINAL_TRAINING_SEED', RUN_SEEDS_LOCAL[0]))
MODEL_WEIGHTS = override('MODEL_WEIGHTS', 'imagenet')
FINAL_VAL_SIZE = float(override('FINAL_VAL_SIZE', 0.15))
FINE_TUNE_FRACTION = resolve_fine_tune_fraction(override('FINE_TUNE_FRACTION', FINE_TUNE_FRACTION))
USE_TRAINING_AUGMENTATION = bool(override('USE_TRAINING_AUGMENTATION', True))
PROCESSED_MANIFEST_PATH = Path(str(override('PROCESSED_MANIFEST_PATH', GENERATED_SPLITS_ROOT / 'processed_manifest.csv')))
OUTPUT_ROOT = Path(str(override('FINAL_DEPLOYMENT_OUTPUT_ROOT', TRAINING_OUTPUTS_ROOT / 'mobilenetv3small_8samples_final_deployment_cnn_only')))

detected_gpu_names = configure_tensorflow_for_training()
if detected_gpu_names:
    print(f'Using GPU(s): {detected_gpu_names}')

written_paths = train_final_deployment_model(PROCESSED_MANIFEST_PATH, OUTPUT_ROOT)
print('\n'.join(f'{name}: {path}' for name, path in written_paths.items()))

finish_notebook_cell_progress(NB_06_TRAIN_FINAL_DEPLOYMENT_MODEL_CELL_PROGRESS_3)


[START] 06_train_final_deployment_model.ipynb | Train final deployment model [0/1 step] elapsed=0.0s
Using GPU(s): ['NVIDIA GeForce RTX 4050 Laptop GPU']
model_path: C:\Users\Adriaan M. Dimate\Desktop\development\school\meatlens-training-2\training_outputs\roboflow\mobilenetv3small_8samples_final_deployment_cnn_only\models\meatlens_final_8samples_cnn_only_mobilenetv3small.keras
predictions_path: C:\Users\Adriaan M. Dimate\Desktop\development\school\meatlens-training-2\training_outputs\roboflow\mobilenetv3small_8samples_final_deployment_cnn_only\final_validation_predictions.csv
history_path: C:\Users\Adriaan M. Dimate\Desktop\development\school\meatlens-training-2\training_outputs\roboflow\mobilenetv3small_8samples_final_deployment_cnn_only\final_training_history.csv
metadata_path: C:\Users\Adriaan M. Dimate\Desktop\development\school\meatlens-training-2\training_outputs\roboflow\mobilenetv3small_8samples_final_deployment_cnn_only\deployment_metadata.json
[RUNNING] 06_train_final_deploy